# Surrogate-1 v2 — Colab T4 (full v18 stack)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arkashira/surrogate-1-harvest/blob/main/notebooks/v2-train-colab.ipynb)

## คลิกแล้วทำตามนี้ — 3 step, ที่เหลือ auto

**Step 1**: คลิกปุ่ม "Open In Colab" ข้างบน (หรือเปิด URL นี้):
```
https://colab.research.google.com/github/arkashira/surrogate-1-harvest/blob/main/notebooks/v2-train-colab.ipynb
```

**Step 2**: ไปที่ **Runtime → Change runtime type → T4 GPU** (ฟรี). กดบันทึก.

**Step 3**: ที่แถบซ้ายของ Colab → ไอคอนรูปกุญแจ 🔑 → Add new secret:
  - Name = `HF_TOKEN`  · Value = (token จาก `~/.note` row `HF_TOKEN` บน Mac, ตัวที่ขึ้นต้น `hf_t...`, user `surrogate1` PRO+admin)
  - กด toggle **Notebook access** ด้านขวา ✓

**Step 4**: **Runtime → Run all** (Ctrl/Cmd + F9). จบ.

Cell 1 = bootstrap, Cell 2 = train (6-10 ชม.), Cell 3 = deploy HF Space (auto). Colab T4 16GB session 12 ชั่วโมง — ครอบคลุม. ถ้า disconnect: เปิด tab เดิม → Run all อีกครั้ง — `SFTTrainer` resume จาก checkpoint อัตโนมัติ ไม่ลบของเก่า.

## Techniques ที่ใช้ (R1-R12 + V13-V18)

Notebook นี้ fetch + run `bin/surrogate-1-train-v18-mission.py` (2900 บรรทัด) จาก repo trunk. ครอบคลุม:
- **R1** LoRA r=32 + all-linear targets · **R2** DoRA · **R5** sample packing (4-8× throughput)
- **R6** NEFTune α=5 · **R8** gradient checkpointing · **R9** AdamW 8-bit paged
- **R11** cosine LR · **R12** multi-source interleave (axentx-pairs A+B+C+D)
- **V13** TRL ≥0.21 (AsyncGRPO/GKDTrainer/DPO) · PEFT ≥0.19 (LoRA-GA) · APOLLO-Mini
- **V14** unified ingest+train (Cerebras/Groq distill → push 9 datasets ก่อน train ถ้ามี token)
- **V15** PiSSA init · **V16** LoRA+ (LR-A vs LR-B ratio) · **V17** spectrum top-50% layers
- **V18** hardware-vs-base sanity (T4 auto-falls FP8→BF16) + adapter naming by base family
- (Optional) post-SFT GRPO RL pass — set `RUN_GRPO=1` ใน Colab Secrets ถ้าอยาก

v18 มี hardware-aware base picker → T4 16GB → auto Qwen2.5-Coder-7B; T4×2 32GB (Kaggle) → 14B; A100/H100 → 32B.

## Output (auto-named)

Adapter จะ push ขึ้น Hub เป็น `axentx/surrogate-1-7B-v1.5-qwen2.5-coder` (auto-suffix ตาม base family/size). Cell 3 ของ notebook deploy HF Space (`ashirato/surrogate-1-v2`) ทันทีหลัง train เสร็จ

In [ ]:
# Cell 1 — bootstrap (env + secrets + fetch v18 script + verify GPU)
#
# ทุก env vars มี default ที่ optimal สำหรับ Colab T4 16GB. Override โดย
# ตั้ง Secret ตามชื่อใน Colab 🔑 panel ถ้าอยาก (เช่น เพิ่ม MAX_SAMPLES,
# เปลี่ยน BASE_MODEL, เปิด RUN_GRPO=1).
import os, subprocess, sys
from google.colab import userdata

# Mandatory token
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
    print('  ✓ HF_TOKEN loaded from Colab Secrets')
except Exception:
    raise SystemExit(
        '\n❌ HF_TOKEN secret missing.\n'
        '   Open the 🔑 panel on the left, add HF_TOKEN, toggle Notebook access, then re-run.'
    )

# Optional secrets — pulled if present, defaulted if not
for k in (
    'WANDB_API_KEY', 'CEREBRAS_API_KEY', 'GROQ_API_KEY',
    'BASE_MODEL', 'HUB_MODEL_ID', 'SUR_LORA_INIT', 'SUR_LORA_PLUS_RATIO',
    'LORA_R', 'MAX_SAMPLES', 'EPOCHS', 'SEQ_LEN', 'LEARNING_RATE',
    'RUN_GRPO', 'MAGPIE_TAKE', 'TAKE_TOOLACE', 'TAKE_MULTIIAC',
    'TAKE_XLAM', 'TAKE_ITBENCH', 'TAKE_CODEFB',
    'DISABLE_AL', 'AL_SAMPLE_CAP', 'SPECTRUM_TOP_FRACTION',
    'SPACE_ID', 'V18_REF',
):
    try:
        v = userdata.get(k)
        if v:
            os.environ[k] = v
    except Exception:
        pass

# Defaults — optimal for Colab T4 16GB single-GPU. Override via Colab Secret.
os.environ.setdefault('BASE_MODEL', 'Qwen/Qwen2.5-Coder-7B-Instruct')
os.environ.setdefault('LORA_R', '32')
os.environ.setdefault('MAX_SAMPLES', '50000')
os.environ.setdefault('EPOCHS', '1')
os.environ.setdefault('SEQ_LEN', '2048')
os.environ.setdefault('LEARNING_RATE', '2e-4')
os.environ.setdefault('SUR_LORA_INIT', 'pissa_niter_4')
os.environ.setdefault('SUR_LORA_PLUS_RATIO', '16.0')
os.environ.setdefault('SPECTRUM_TOP_FRACTION', '0.5')
os.environ.setdefault('RUN_GRPO', '0')
os.environ.setdefault('SPACE_ID', 'ashirato/surrogate-1-v2')
os.environ.setdefault('V18_REF', 'main')

# Verify GPU
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free,compute_cap',
                '--format=csv'], check=False)
import torch
print(f'\ncuda: {torch.cuda.is_available()}  device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f'compute capability: {cap[0]}.{cap[1]} (T4=7.5, A100=8.0, H100=9.0)')

# Fetch v18 script (always tip; pin via V18_REF Secret if needed)
ref = os.environ['V18_REF']
url = f'https://raw.githubusercontent.com/arkashira/surrogate-1-harvest/{ref}/bin/surrogate-1-train-v18-mission.py'
subprocess.run(['curl', '-sSf', '-o', '/content/v18_train.py', url], check=True)
n_lines = int(subprocess.check_output(['wc', '-l', '/content/v18_train.py']).split()[0])
print(f'\n  ✓ fetched v18 ({n_lines} lines) from ref={ref}')

# Print effective config
print('\nv18 config (override via Colab Secrets if you want):')
for k in ('BASE_MODEL', 'LORA_R', 'MAX_SAMPLES', 'EPOCHS', 'SEQ_LEN', 'LEARNING_RATE',
         'SUR_LORA_INIT', 'SUR_LORA_PLUS_RATIO', 'SPECTRUM_TOP_FRACTION', 'RUN_GRPO',
         'SPACE_ID'):
    print(f'  {k}={os.environ.get(k, "(unset)")}')


In [ ]:
# Cell 2 — train (v18 stack, 6-10h on T4 with 50k samples)
#
# v18 ทำเอง:
#  1. install deps (transformers ≥4.55, TRL ≥0.21, PEFT ≥0.19, bitsandbytes,
#     accelerate, deepspeed, triton, liger-kernel, apollo-torch — opt-in)
#  2. (optional) Phase -1 distill via Cerebras/Groq → push 9 axentx datasets
#  3. hardware-aware base pick (T4 16GB → 7B)
#  4. Build BNB 4-bit + LoRA r=32 + DoRA + PiSSA init + LoRA+ + Spectrum
#  5. SFTTrainer with packing=True, NEFTune α=5, gradient_checkpointing,
#     8-bit paged AdamW, cosine LR + 3% warmup
#  6. (optional, RUN_GRPO=1) GRPO RL post-pass
#  7. push adapter to axentx/surrogate-1-{SIZE}B-v1.5-{base-family}
#
# Auto-resumes from checkpoint if disconnected (re-run this cell).
%cd /content
%run /content/v18_train.py


In [ ]:
# Cell 3 — deploy HF Space v2 (ใช้ adapter ที่ Cell 2 เพิ่ง push)
#
# Targets ashirato/surrogate-1-v2 (PRO + ZeroGPU eligible). Idempotent —
# safe to re-run. ถ้า Space ยังไม่มี: create. ถ้ามีอยู่: upload + restart.
import os, subprocess
ref = os.environ['V18_REF']
subprocess.run(['mkdir', '-p', '/content/hf-space-v2'], check=True)
for fname in ('app.py', 'requirements.txt', 'README.md', '.gitattributes', 'deploy.sh'):
    url = f'https://raw.githubusercontent.com/arkashira/surrogate-1-harvest/{ref}/hf-space-v2/{fname}'
    subprocess.run(['curl', '-sSf', '-o', f'/content/hf-space-v2/{fname}', url], check=True)
subprocess.run(['chmod', '+x', '/content/hf-space-v2/deploy.sh'], check=True)

# HF_TOKEN ที่ user ให้คือ surrogate1 admin — แต่ deploy.sh ต้องใช้ token
# ที่เป็น **owner ของ ashirato namespace** เพื่อให้ Space ได้ ZeroGPU ฟรี.
# ถ้ามี Colab Secret ชื่อ HF_TOKEN_PRO_WRITE จะใช้อันนั้น; ถ้าไม่มี fallback
# ไป HF_TOKEN (Space จะอยู่บน namespace ของ token owner แทน).
from google.colab import userdata
try:
    pro_tok = userdata.get('HF_TOKEN_PRO_WRITE')
    if pro_tok:
        os.environ['HF_TOKEN'] = pro_tok
        print('  ✓ using HF_TOKEN_PRO_WRITE (ashirato/* namespace, ZeroGPU eligible)')
except Exception:
    pass

subprocess.run(['bash', '/content/hf-space-v2/deploy.sh'], check=True)
print(f'\n✓ Space live at https://huggingface.co/spaces/{os.environ["SPACE_ID"]}')
